# Prompt comparison

Checking whether v2/v3 prompts actually beat the v1 one that's currently in `server/`. Metrics: ROUGE + BERTScore on the summary, F1 on action items, and an LLM judge.

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.precision", 3)

Load `results/results.csv` (run `run_eval.py` first). If it's missing, run the harness inline without BERTScore so it's quick.

In [ ]:
csv_path = "results/results.csv"
if not os.path.exists(csv_path):
    from run_eval import evaluate
    df = evaluate(use_bertscore=False)
    os.makedirs("results", exist_ok=True)
    df.to_csv(csv_path, index=False)
else:
    df = pd.read_csv(csv_path)

df

Mean of each metric per prompt version.

In [ ]:
numeric_cols = df.select_dtypes("number").columns
summary = df.groupby("prompt_version")[numeric_cols].mean().round(3)
summary

In [ ]:
# baseline vs best on the two metrics I care about
for col in ["ai_f1", "judge_faithfulness"]:
    if col in summary.columns:
        base = summary.loc["v1_baseline", col]
        best_v = summary[col].idxmax()
        print(f"{col}: {base:.3f} (v1) -> {summary.loc[best_v, col]:.3f} ({best_v})")

### Action items

This is the one that actually matters — catching real action items without making any up.

In [ ]:
ai = summary[[c for c in ["ai_precision", "ai_recall", "ai_f1"] if c in summary.columns]]
ax = ai.plot(kind="bar", figsize=(8, 4.5), rot=0, colormap="viridis")
ax.set_title("Action-item extraction")
ax.set_ylim(0, 1)
ax.legend(loc="lower right")
plt.tight_layout()
plt.show()

### Summary similarity (ROUGE / BERTScore)

In [ ]:
sim_cols = [c for c in ["rouge1", "rouge2", "rougeL", "bertscore_f1"] if c in summary.columns]
ax = summary[sim_cols].plot(kind="bar", figsize=(8, 4.5), rot=0, colormap="mako")
ax.set_title("Executive summary vs reference")
ax.set_ylim(0, 1)
plt.tight_layout()
plt.show()

### Judge scores

In [ ]:
judge_cols = [c for c in ["judge_faithfulness", "judge_completeness", "judge_hallucination_free"] if c in summary.columns]
if judge_cols:
    ax = summary[judge_cols].plot(kind="bar", figsize=(8, 4.5), rot=0, colormap="flare")
    ax.set_title("LLM-as-judge (1-5)")
    ax.set_ylim(0, 5)
    plt.tight_layout()
    plt.show()
else:
    print("no judge columns - ran with --no-judge")

Per-meeting F1, to see which meetings are hard rather than just the average.

In [ ]:
if "ai_f1" in df.columns:
    pivot = df.pivot_table(index="meeting_id", columns="prompt_version", values="ai_f1")
    plt.figure(figsize=(7, 4))
    sns.heatmap(pivot, annot=True, fmt=".2f", cmap="YlGnBu", vmin=0, vmax=1)
    plt.title("action-item F1 by meeting")
    plt.tight_layout()
    plt.show()

**Notes:** _(fill in after a run)_